<a href="https://colab.research.google.com/github/Francja/geneticAlgorithm/blob/master/Genetic_algorithm_for_DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Genetic algorithm search over DNN architectures

This notebook drives the `dnn_ga` package (see `src/dnn_ga/`) instead of reimplementing the genetic algorithm inline. The original inline version of this notebook had a real syntax error on its last line (`final_text(pop_bag, text_summary))` - wrong argument and an unmatched `)`) and could not run as-is; it also used the now-deprecated `tf.estimator.DNNClassifier`, aliased genomes during selection (a correctness bug, not just a style issue), never returned the winning architecture, had no elitism, only ever mutated one layer's width, and compared floats for exact equality to detect stagnation. All of that is fixed in the package and covered by `tests/`; see `README.md` for the full list.


In [ ]:
# Uncomment if running in a fresh Colab runtime:
# !pip install -q -e .


In [ ]:
from random import Random

from dnn_ga.config import GAConfig
from dnn_ga.fitness import load_iris_dataset, make_keras_dnn_cost
from dnn_ga.ga import run_ga
from dnn_ga.logging_utils import configure_logging, log_generation
from dnn_ga.presets import DNN_PRESET


## Configuration

Same parameters as the original notebook (`MIN_LAYER`/`MAX_LAYER`/`MAX_NEURONS`/`POPULATION`/`GENERATIONS`/`TARGET_SCORE`/`MUTATION_CHANCE`/`NO_EVOLVE_GENS`), plus `STEPS` (now `epochs`, since training moved from `tf.estimator` to `tf.keras`), `elitism_count` (new - keeps the best architecture found so far from being lost), and `structural_mutation_chance` (new - lets mutation add/remove a layer, not just tweak one layer's width).

In [ ]:
STEPS = 350  # epochs for tf.keras training, one fitness evaluation per genome

# DNN_PRESET is the single source of truth for these hyperparameters, shared
# with the CLI's --mode dnn (src/dnn_ga/cli.py) so the two can't drift apart.
cfg = GAConfig(**DNN_PRESET)


## Data + fitness function

`make_keras_dnn_cost` builds a fitness function that trains a `tf.keras.Sequential` MLP per genome and returns held-out accuracy. This replaces `tf.estimator.DNNClassifier`, which is deprecated in TensorFlow.

In [ ]:
train_x, train_y, test_x, test_y = load_iris_dataset()
cost_fn = make_keras_dnn_cost(train_x, train_y, test_x, test_y, epochs=STEPS, verbose=0)


## Logging

Logs to the console and, optionally, to a file. In the original notebook this was hardcoded to a Google Drive path (`/content/drive/My Drive/...`) via manual `open(filename, 'a')` calls, which only worked inside Colab with Drive mounted and crashed the whole run on any I/O error. Here the path is just a parameter - point it at a mounted Drive path if you're in Colab and want that, or leave it local.

In [ ]:
from datetime import datetime

# In Colab, uncomment to persist logs to Drive instead of the local runtime disk:
# from google.colab import drive
# drive.mount('/content/drive')
# log_dir = '/content/drive/My Drive/Inzynierka'
log_dir = '.'

log_path = f"{log_dir}/AG_NN_Output_{datetime.now():%Y-%m-%d_%H-%M-%S}.log"
logger = configure_logging(log_path)
logger.info("Config: %s", cfg)


## Run

`run_ga` returns a `GAResult` carrying the winning **architecture** (`best_individual.genome`), not just its score - the original code only ever printed a numeric best score, so recovering which network achieved it meant reading the printed log by eye.

In [ ]:
rng = Random(0)  # set to None for a fresh random run each time

def on_generation(gen, population, best):
    log_generation(logger, gen, population, best)

result = run_ga(cfg, cost_fn, rng=rng, on_generation=on_generation)


In [ ]:
logger.info("Stopped after %d generation(s): %s", result.generations_run, result.stop_reason)
logger.info("Best architecture: %s (accuracy=%.4f)", result.best_individual.genome, result.best_individual.score)

result.best_individual.genome, result.best_individual.score
